# Importaciones necesarias

In [2]:
!pip install torch
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ----- ---------------------------------- 16.0/122.1 MB 95.6 MB/s eta 0:00:02
   ----------- ---------------------------- 33.8/122.1 MB 86.6 MB/s eta 0:00:02
   ------------------- -------------------- 59.0/122.1 MB 97.4 MB/s eta 0:00:01
   -------------------- ------------------- 61.1/122.1 MB 81.6 MB/s eta 0:00:01
   ------------------------- -------------- 77.6/122.1 MB 75.8 MB/s eta 0:00:01
   -------------------------------- ------- 99.6/122.1 MB 81.4 MB/s eta 0:00:01
   ------------------------------------- - 116.4/122.1 MB 84.8 MB/s eta 0:00:01
   ------------------------------------- - 118.2/122.1 MB 71.9 MB/s eta 0:00:01
   --------------------------------------  121.9/122.1 MB 72.8 MB/s eta 0:00:01
   --------------------------------------  121.9/122.1 MB 72.8 MB/s eta 0:00:01
   --------------------------------------  121.9/122.1 MB 72.8 MB/s eta 0:00:01
   ---------------------------------------- 122.1

# Carga de datos
Tras las importaciones el primer paso necesario saerá importar los datos desde el archivo .csv

In [ ]:
dataset = pd.read_csv('/workspace/student_scores.csv')

print(dataset.head())

   Hours  Scores
0    2.5      21
1    5.1      47
2    3.2      27
3    8.5      75
4    3.5      30


# Preparar el dataset para la regresión
Aquí separaremos las características y el objetivo.

In [4]:
X = dataset[['Hours']].values
y = dataset[['Scores']].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Tamaño de entrenamiento: {X_train.shape[0]} muestras")
print(f"Tamaño de prueba: {X_test.shape[0]} muestras")

Tamaño de entrenamiento: 18 muestras
Tamaño de prueba: 7 muestras


# Construcción de la red neuronal
Se define la red neuronal con una capa de entrada, dos capas ocultas (de 64 y 32 neuronas cada una) y una de salida. 

In [5]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

class StudentNet(nn.Module):
    def __init__(self):
        super(StudentNet, self).__init__()
        self.hidden1 = nn.Linear(1, 64)
        self.relu1 = nn.ReLU()
        self.hidden2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.output = nn.Linear(32, 1)

    def forward(self, x):
        x = self.relu1(self.hidden1(x))
        x = self.relu2(self.hidden2(x))
        x = self.output(x)
        return x

model = StudentNet()
print(model)

StudentNet(
  (hidden1): Linear(in_features=1, out_features=64, bias=True)
  (relu1): ReLU()
  (hidden2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (output): Linear(in_features=32, out_features=1, bias=True)
)


# Entrenamiento de la red neuronal
En este paso vamos a configurar la función de pérdida (MSE), el optimizador (Adam) y entrenaremos la red durante 200 épocas. 

In [6]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 200

for epoch in range(epochs):
    y_pred = model(X_train_tensor)
    
    loss = criterion(y_pred, y_train_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f'Época [{epoch+1}/{epochs}], Pérdida (MSE): {loss.item():.4f}')

Época [20/200], Pérdida (MSE): 88.5407
Época [40/200], Pérdida (MSE): 92.9710
Época [60/200], Pérdida (MSE): 32.9830
Época [80/200], Pérdida (MSE): 31.7582
Época [100/200], Pérdida (MSE): 30.5504
Época [120/200], Pérdida (MSE): 30.2082
Época [140/200], Pérdida (MSE): 29.8520
Época [160/200], Pérdida (MSE): 29.5143
Época [180/200], Pérdida (MSE): 29.1893
Época [200/200], Pérdida (MSE): 28.8690


# Evaluación del modelo
Por último vamos a evaluar el modelo y realizamos las predicciones con el conjunto de prueba (sin calcular gradientes para ahorrar memoria) y aplicamos las métricas de scikit-learn.

In [7]:
model.eval()

with torch.no_grad():
    y_test_pred = model(X_test_tensor)

y_test_pred_np = y_test_pred.numpy()
y_test_np = y_test_tensor.numpy()

r2 = r2_score(y_test_np, y_test_pred_np)
mae = mean_absolute_error(y_test_np, y_test_pred_np)
rmse = np.sqrt(mean_squared_error(y_test_np, y_test_pred_np))

print("\n--- Resultados de la Evaluación ---")
print(f"R2 (Coeficiente de determinación): {r2:.4f}")
print(f"MAE (Error absoluto medio): {mae:.4f}")
print(f"RMSE (Raíz del error cuadrático medio): {rmse:.4f}")


--- Resultados de la Evaluación ---
R2 (Coeficiente de determinación): 0.9498
MAE (Error absoluto medio): 4.6030
RMSE (Raíz del error cuadrático medio): 5.1431
